In [ ]:
#Only run once to install
!pip install statsmodels pmdarima

  Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached pmdarima-2.1.1-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached cython-3.2.4-cp314-cp314-win_amd64.whl.metadata (7.7 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl (9.6 MB)
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ------------- -------------------------- 4.2/12.4 MB 26.0 MB/s eta 0:00:01
   ------------------------ --------------- 7.6/12.4 MB 30.0 MB/s eta 0:00:01
   -----


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Predicted Count of Crashes per Month in 2026

In [ ]:
"""
forecast_crash_frequency.py
----------------------------
Uses SARIMA to forecast monthly crash counts for 2026,
based on historical crash-level data with a date column.
 
Requirements:
    pip install pandas statsmodels pmdarima
"""
 
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

In [ ]:
#CONFIGURATION
INPUT_FILE   = "atx_crash_data_2018-2026_clean.csv"   # Path to your crash-level data
DATE_COLUMN  = "Crash timestamp (US/Central)"       # Name of the date column in your data
DATE_FORMAT  = "%m/%d/%Y %H:%M"         # Adjust if your dates are in a different format
OUTPUT_FILE  = "outputs/crash_forecast_2026.csv"
 
# SARIMA order — set USE_AUTO_ARIMA = True to detect these automatically,
# or set USE_AUTO_ARIMA = False and specify (p, d, q) and (P, D, Q, s) manually.
USE_AUTO_ARIMA = True
 
# Manual SARIMA orders (only used if USE_AUTO_ARIMA = False)
ORDER          = (1, 1, 1)        # (p, d, q)  — non-seasonal component
SEASONAL_ORDER = (1, 1, 1, 12)   # (P, D, Q, s) — seasonal component (s=12 for monthly)

In [ ]:
#LOAD & AGGREGATE DATA
df = pd.read_csv(INPUT_FILE, parse_dates=[DATE_COLUMN])
 
# Drop rows with missing dates
df = df.dropna(subset=[DATE_COLUMN])
 
# Aggregate to monthly crash counts
monthly = (
    df.set_index(DATE_COLUMN)
    .resample("MS")           # "MS" = Month Start frequency
    .size()
    .rename("crash_count")
    .reset_index()
    .rename(columns={DATE_COLUMN: "month"})
)
 
print(f"  Date range: {monthly['month'].min().date()} → {monthly['month'].max().date()}")
print(f"  Total months: {len(monthly)}")
print(f"  Avg crashes/month: {monthly['crash_count'].mean():.1f}")
 
# Set month as index for time series modeling
ts = monthly.set_index("month")["crash_count"].asfreq("MS")

  Date range: 2018-01-01 → 2026-01-01
  Total months: 97
  Avg crashes/month: 561.2


In [ ]:
#FIT SARIMA MODEL
 
if USE_AUTO_ARIMA:
    print("\nRunning auto_arima to find best SARIMA order (this may take a moment)...")
    auto_model = pm.auto_arima(
        ts,
        seasonal=True,
        m=12,                 # Monthly seasonality
        stepwise=True,        
        suppress_warnings=True,
        error_action="ignore",
        information_criterion="aic",
    )
    order          = auto_model.order
    seasonal_order = auto_model.seasonal_order
    print(f"  Best order:          ARIMA{order}")
    print(f"  Best seasonal order: {seasonal_order}")
else:
    order          = ORDER
    seasonal_order = SEASONAL_ORDER
    print(f"\nUsing manual SARIMA order:  ARIMA{order}x{seasonal_order}")
 
print("\nFitting final SARIMA model...")
model = SARIMAX(
    ts,
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fitted = model.fit(disp=False)
print(f"  AIC: {fitted.aic:.2f}  |  BIC: {fitted.bic:.2f}")


Running auto_arima to find best SARIMA order (this may take a moment)...
  Best order:          ARIMA(1, 1, 0)
  Best seasonal order: (0, 0, 0, 12)

Fitting final SARIMA model...
  AIC: 1079.98  |  BIC: 1085.08


In [ ]:
#FORECAST 2026
 
print("\nForecasting Jan–Dec 2026...")
 
# Number of steps from end of training data to end of 2026
last_train_month = ts.index[-1]
forecast_end     = pd.Timestamp("2026-12-01")
steps            = (
    (forecast_end.year - last_train_month.year) * 12
    + (forecast_end.month - last_train_month.month)
)

# If training data already covers some 2026 months, forecast start is after last training month
forecast_start = last_train_month + pd.DateOffset(months=1)
 
forecast_result = fitted.get_forecast(steps=steps)
forecast_mean   = forecast_result.predicted_mean
forecast_ci     = forecast_result.conf_int(alpha=0.05)   # 95% CI
 
# Slice to only 2026 months
forecast_2026 = forecast_mean[(forecast_mean.index >= forecast_start) & (forecast_mean.index <= "2026-12-01")]
forecast_ci_2026 = forecast_ci[(forecast_ci.index >= forecast_start) & (forecast_ci.index <= "2026-12-01")]
 
# Round to whole crashes and clip negatives to 0
forecast_2026_rounded = forecast_2026.round().clip(lower=0).astype(int)


Forecasting Jan–Dec 2026...


In [ ]:
#BUILD OUTPUT
 
output = pd.DataFrame({
    "month":                  forecast_2026_rounded.index.strftime("%Y-%m"),
    "forecasted_crash_count": forecast_2026_rounded.values,
})

#Since training data goes to January 2026, I am adding the the true value for January 2026 for our data for January. The model is only forecasting Feb 2026- Dec 2026.
jan_actual = pd.DataFrame({
    "month": ["2026-01"],
    "forecasted_crash_count": [int(ts["2026-01-01"])]  # actual value from training data
})

output = pd.concat([jan_actual, output], ignore_index=True)
 
output.to_csv(OUTPUT_FILE, index=False)
 
print(f"\n{'─'*45}")
print(f"  2026 Monthly Crash Frequency Forecast")
print(f"{'─'*45}")
print(output.to_string(index=False))
print(f"{'─'*45}")
print(f"  Annual total: {output['forecasted_crash_count'].sum():,} crashes")
print(f"\nSaved to: {OUTPUT_FILE}")


─────────────────────────────────────────────
  2026 Monthly Crash Frequency Forecast
─────────────────────────────────────────────
  month  forecasted_crash_count
2026-01                     174
2026-02                     273
2026-03                     234
2026-04                     249
2026-05                     243
2026-06                     246
2026-07                     245
2026-08                     245
2026-09                     245
2026-10                     245
2026-11                     245
2026-12                     245
─────────────────────────────────────────────
  Annual total: 2,889 crashes

Saved to: outputs/crash_forecast_2026.csv


## Forecast Crash Costs

In [49]:
"""
forecast_crash_costs.py
------------------------
Uses an existing Random Forest model to predict crash costs for 2026.
For each month, bootstraps a sample of historical crashes from that
calendar month, scores them with the RF, then combines with the SARIMA
frequency forecast to produce total monthly cost estimates.
 
Requirements:
    pip install pandas numpy scikit-learn
"""
 
import pickle
import pandas as pd
import numpy as np

In [ ]:
#CONFIGURATION
 
CRASH_FILE       = "atx_crash_data_2018-2026_clean.csv"            # Historical crash-level data
DATE_COLUMN  = "Crash timestamp (US/Central)"       # Name of the date column in your data
DATE_FORMAT  = "%m/%d/%Y %H:%M"         # Adjust if your dates are in a different format
FREQUENCY_FILE   = "outputs/crash_forecast_2026.csv"  # Output from forecast_crash_frequency.py
MODEL_FILE       = "outputs/best_rf_model.pkl"    # Path to your saved RF model
 
# Columns used as features for the RF model
FEATURE_COLUMNS  = [
    "crash_speed_limit"
    ,"road_constr_zone_fl"
    ,"latitude"
    ,"longitude"
    ,"onsys_fl"
    ,"private_dr_fl"
    ,"Location group"
    ,"day_of_week"
    ,"week_of_year"
    ,"hour_of_day"
    ,"micromobility device_involved"
    ,"train_involved"
    ,"e-scooter_involved"
    ,"pedestrian_involved"
    ,"motorcycle_involved"
    ,"large passenger vehicle_involved"
    ,"bicycle_involved"
    ,"passenger car_involved"
    ,"motor vehicle – other_involved"
]
 
OUTPUT_FILE      = "outputs/cost_forecast_2026.csv"
 
# Bootstrap settings
N_BOOTSTRAP      = 1000    # Number of crashes to sample per month
RANDOM_SEED      = 42      # For reproducibility

In [ ]:
#LOAD INPUTS
 
print("Loading model and data...")
 
# Load trained Random Forest
with open(MODEL_FILE, "rb") as f:
    rf_model = pickle.load(f)
 
# Load historical crash data
crashes = pd.read_csv(CRASH_FILE, parse_dates=[DATE_COLUMN])
crashes = crashes.dropna(subset=[DATE_COLUMN])
crashes["calendar_month"] = crashes[DATE_COLUMN].dt.month   # 1–12
 
# Load 2026 frequency forecast
freq_df = pd.read_csv(FREQUENCY_FILE)
freq_df["month_dt"] = pd.to_datetime(freq_df["month"])
freq_df["calendar_month"] = freq_df["month_dt"].dt.month
 
print(f"  Historical crashes loaded:  {len(crashes):,} rows")
print(f"  Forecast months loaded:     {len(freq_df)}")
print(f"  RF features expected:       {FEATURE_COLUMNS}")

Loading model and data...
  Historical crashes loaded:  54,434 rows
  Forecast months loaded:     12
  RF features expected:       ['crash_speed_limit', 'road_constr_zone_fl', 'latitude', 'longitude', 'onsys_fl', 'private_dr_fl', 'Location group', 'day_of_week', 'week_of_year', 'hour_of_day', 'micromobility device_involved', 'train_involved', 'e-scooter_involved', 'pedestrian_involved', 'motorcycle_involved', 'large passenger vehicle_involved', 'bicycle_involved', 'passenger car_involved', 'motor vehicle – other_involved']


In [52]:
crashes.head()

,ID,Crash ID,crash_fatal_fl,case_id,rpt_block_num,rpt_street_name,rpt_street_sfx,crash_speed_limit,road_constr_zone_fl,latitude,...,micromobility device_involved,train_involved,e-scooter_involved,pedestrian_involved,motorcycle_involved,large passenger vehicle_involved,bicycle_involved,passenger car_involved,motor vehicle – other_involved,calendar_month
0,92807,16839445.0,False,190080930,1200,MOPAC,EXPY,65.0,False,30.288379,...,False,False,False,False,False,True,False,True,False,1
1,93467,16854755.0,False,190151441,1700,OHLEN,RD,35.0,False,30.363175,...,False,False,False,False,False,False,False,True,False,1
2,93480,16854819.0,False,190160293,3100,IH 35,HWY,60.0,False,30.234100,...,False,False,False,False,False,True,False,False,False,1
3,94289,16870096.0,False,190260764,300,E PARMER,LN,50.0,False,30.405288,...,False,False,False,False,False,True,False,False,False,1
4,93948,16863200.0,False,190230902,8900,GALEWOOD,DR,25.0,False,30.366156,...,False,False,False,False,False,True,False,True,False,1


In [ ]:
#BOOTSTRAP & SCORE
print("\nBootstrapping and scoring crashes by month...")
 
rng = np.random.default_rng(RANDOM_SEED)
results = []
 
for _, row in freq_df.iterrows():
    month_label   = row["month"]           # e.g. "2026-02"
    cal_month     = row["calendar_month"]  # e.g. 2 for February
    crash_count   = row["forecasted_crash_count"]
 
    # Pull historical crashes from the same calendar month
    month_pool = crashes[crashes["calendar_month"] == cal_month]
 
    if len(month_pool) == 0:
        print(f"  WARNING: No historical data for calendar month {cal_month} — skipping.")
        continue
 
    # Bootstrap sample (with replacement) of size N_BOOTSTRAP
    sample = month_pool[FEATURE_COLUMNS].sample(
        n=N_BOOTSTRAP,
        replace=True,
        random_state=int(rng.integers(0, 99999)),
    )
 
    # Predict cost for each sampled crash
    predicted_costs = rf_model.predict(sample)
 
    # Mean predicted cost per crash for this month
    mean_cost = predicted_costs.mean()
 
    # Total expected cost = mean cost per crash × forecasted crash count
    total_cost = mean_cost * crash_count
 
    # Store per-crash predictions with their month label
    for cost in predicted_costs:
        results.append({
            "month":               month_label,
            "predicted_cost":      round(cost, 2),
        })
 
    print(f"  {month_label}  |  crashes: {crash_count:>4}  |  "
          f"mean cost: ${mean_cost:>10,.2f}  |  total cost: ${total_cost:>14,.2f}")


Bootstrapping and scoring crashes by month...
  2026-01  |  crashes:  174  |  mean cost: $307,147.37  |  total cost: $ 53,443,642.55
  2026-02  |  crashes:  273  |  mean cost: $310,795.78  |  total cost: $ 84,847,249.28
  2026-03  |  crashes:  234  |  mean cost: $314,608.74  |  total cost: $ 73,618,445.92
  2026-04  |  crashes:  249  |  mean cost: $323,504.07  |  total cost: $ 80,552,513.16
  2026-05  |  crashes:  243  |  mean cost: $311,683.67  |  total cost: $ 75,739,132.78
  2026-06  |  crashes:  246  |  mean cost: $330,531.14  |  total cost: $ 81,310,660.12
  2026-07  |  crashes:  245  |  mean cost: $324,390.40  |  total cost: $ 79,475,647.96
  2026-08  |  crashes:  245  |  mean cost: $322,604.45  |  total cost: $ 79,038,089.29
  2026-09  |  crashes:  245  |  mean cost: $331,142.54  |  total cost: $ 81,129,921.78
  2026-10  |  crashes:  245  |  mean cost: $329,632.41  |  total cost: $ 80,759,941.29
  2026-11  |  crashes:  245  |  mean cost: $307,305.04  |  total cost: $ 75,289,734

In [ ]:
#AGGREGATE TO MONTHLY SUMMARY
 
print("\nAggregating to monthly summary...")
 
per_crash_df = pd.DataFrame(results)
 
# Monthly mean cost (from bootstrap predictions)
monthly_mean = (
    per_crash_df.groupby("month")["predicted_cost"]
    .mean()
    .round(2)
    .rename("mean_predicted_cost_per_crash")
)
 
# Re-join with frequency forecast to compute total cost
summary = freq_df[["month", "forecasted_crash_count"]].copy()
summary = summary.join(monthly_mean, on="month")
summary["total_predicted_cost"] = (
    summary["mean_predicted_cost_per_crash"] * summary["forecasted_crash_count"]
).round(2)


Aggregating to monthly summary...


In [ ]:
#SAVE OUTPUTS
 
# Per-crash predictions
per_crash_output = OUTPUT_FILE.replace(".csv", "_per_crash.csv")
per_crash_df.to_csv(per_crash_output, index=False)
 
# Monthly summary
summary.to_csv(OUTPUT_FILE, index=False)
 
print(f"\n{'─'*65}")
print(f"  2026 Monthly Cost Forecast")
print(f"{'─'*65}")
print(summary.to_string(index=False))
print(f"{'─'*65}")
print(f"  Total forecasted cost (2026): ${summary['total_predicted_cost'].sum():,.2f}")
print(f"\nSaved summary to:    {OUTPUT_FILE}")
print(f"Saved per-crash to:  {per_crash_output}")


─────────────────────────────────────────────────────────────────
  2026 Monthly Cost Forecast
─────────────────────────────────────────────────────────────────
  month  forecasted_crash_count  mean_predicted_cost_per_crash  total_predicted_cost
2026-01                     174                      307147.37           53443642.38
2026-02                     273                      310795.78           84847247.94
2026-03                     234                      314608.74           73618445.16
2026-04                     249                      323504.07           80552513.43
2026-05                     243                      311683.67           75739131.81
2026-06                     246                      330531.14           81310660.44
2026-07                     245                      324390.40           79475648.00
2026-08                     245                      322604.45           79038090.25
2026-09                     245                      331142.54           